# PII Detector Comparison

Benchmark **9 PII detectors** — Skyflow Detect, OPF, default GLiNER + 3 GLiNER variants (Nvidia, Gretel small/large), Microsoft Presidio, the ai4privacy ModernBERT model, and the OpenMed PII family — against a sample from any of 5 ai4privacy datasets. Same harness as `eval/src/opf_eval/` — exposed via a notebook for easy hosted runs.

**What this notebook does:**
1. Installs the comparison harness + open-weight detectors (OPF, GLiNER family, ai4privacy, OpenMed) and Presidio + spaCy models
2. Materializes a deterministic fixture set from your chosen dataset
3. Runs each detector you select against the fixtures, writing `raw_<detector>.jsonl` per detector
4. Renders a Markdown report (fair view + raw view + per-language SemEval Type F1)
5. Optional: bar charts of per-category F1, latency comparison

**You'll need:**
- For the local detectors (OPF / GLiNER family / Presidio / ai4privacy / OpenMed): Colab free tier is enough (CPU works; GPU optional)
- For Skyflow Detect: a vault URL, vault ID, and bearer token (set as Colab secrets)

**Estimated wall time** at default config (100 examples, 3 local detectors): ~5 minutes. Adding all 9 detectors at 1k examples is closer to ~40 minutes (OpenMed loads a separate model per language; Nvidia + Gretel large download ~2 GB each).

## 1. Setup — install the harness and dependencies

Run this once per Colab session. Installs:
- The OPF source repo (open-weight PII detector from OpenAI)
- The comparison harness (this repo's `eval/` package)
- GLiNER, Presidio, and supporting libs

**Edit `HARNESS_REPO` below** to point at the fork/branch where this notebook's source lives.

In [ ]:
# spaCy models for Presidio. en_core_web_lg is required; the others are
# optional and only needed if you want multilingual Presidio (skip if running
# default English-only Presidio — most cases).
#
# IMPORTANT (Colab): running this cell upgrades numpy/typing-extensions and
# Colab will pop up "RESTART SESSION" when it finishes. Click restart, then
# use Runtime → Run all to continue. Doing the spaCy install first means the
# slow harness install in the next cell only happens once (after the restart),
# rather than getting wiped out and re-run.
!python -m spacy download en_core_web_lg -q

# Uncomment the next 5 lines for multilingual Presidio (~3 GB extra download):
# !python -m spacy download nl_core_news_lg -q
# !python -m spacy download fr_core_news_lg -q
# !python -m spacy download de_core_news_lg -q
# !python -m spacy download it_core_news_lg -q
# !python -m spacy download es_core_news_lg -q

print("\nspaCy models ready. If Colab prompts to restart the session, do it now and re-run from the top.")


In [ ]:
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
HARNESS_BRANCH = "main"   # set to a branch name (e.g. "jstjoe/notebook-iteration") to test a PR before merge
OPF_REPO = "https://github.com/openai/privacy-filter.git"

import os, subprocess, sys

# Triton has no stable Apple Silicon support and isn't needed on Colab CPU/GPU.
# Setting before any opf import keeps the runtime on the vanilla PyTorch MoE path.
os.environ.setdefault("OPF_MOE_TRITON", "0")

PIP = f"{sys.executable} -m pip"  # ensure we install into the notebook's kernel

def _run(cmd, *, msg, show_output=False):
    print(f"==> {msg}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 or show_output:
        if r.stdout: print(r.stdout)
        if r.stderr: print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{msg} failed (exit {r.returncode}). See output above.")

if not os.path.exists("/content/privacy-filter"):
    _run(f"git clone --depth 1 {OPF_REPO} /content/privacy-filter", msg="clone privacy-filter")
if not os.path.exists("/content/local-privacy"):
    _run(f"git clone --depth 1 --branch {HARNESS_BRANCH} {HARNESS_REPO} /content/local-privacy", msg=f"clone local-privacy@{HARNESS_BRANCH} (must be public, or use a PAT in HARNESS_REPO)")

# Pip-install for dependency resolution (torch, datasets, gliner, presidio, etc).
# Whether or not pip places the top-level packages in site-packages reliably on
# Colab, we also add the source directories to sys.path below so imports always
# work.
_run(f"{PIP} install -q /content/privacy-filter", msg="install opf + deps")
_run(f"{PIP} install -q /content/local-privacy/eval", msg="install opf-eval + deps")
_run(f"{PIP} install -q /content/local-privacy/api", msg="install opf-api + deps (needed for Sanitization → label_token mode)")

# Belt-and-suspenders: expose the source trees on sys.path so `import opf` and
# `import opf_eval` resolve regardless of what pip did with the editable hooks.
for src_dir in ("/content/privacy-filter", "/content/local-privacy/eval/src", "/content/local-privacy/api/src"):
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

# Confirm the kernel can import everything we need.
import importlib
for mod in ("opf", "opf_eval", "opf_eval.runner", "opf_eval.report", "opf_eval.fixtures", "opf_eval.transforms", "opf_api", "opf_api.vault_tokens"):
    importlib.import_module(mod)
    print(f"  ok: {mod}")

print("\nsetup complete.")


## 2. Imports and configuration

Tweak the config cell to change sample size, dataset, detectors, and output location.

**Datasets:**
- `pii_masking_300k` (default), `pii_masking_200k`, `pii_masking_400k` — legacy ai4privacy variants (different vocabularies)
- `openpii_nano` (1k), `openpii_mini` (10k) — OpenPII vocabulary

**Detector names:**
- `opf` — OpenAI Privacy Filter (open-weight, local) — overall local leader
- `gliner` — default GLiNER multilingual PII (`urchade/gliner_multi_pii-v1`) — prompts auto-restricted to dataset vocab
- `gliner_nvidia` — Nvidia gliner-PII on `urchade/gliner_large-v2.1` (570M base, threshold 0.3, NVIDIA Open Model License) — strongest of the GLiNER variants
- `gliner_gretel_small` / `gliner_gretel_large` — Gretel bi-encoder GLiNER (threshold 0.7, English-only training, snake_case 41-label vocab)
- `ai4privacy_modernbert` — ai4privacy ModernBERT-base (~150M, MIT, 8 languages, OpenPII vocab)
- `openmed` — OpenMed PII via `openmed.extract_pii(lang=…)` — DeBERTa-based per-language models, snake_case 55-label vocab
- `presidio` — Microsoft Presidio English-only (regex + NER, local)
- `presidio_multilang` — Presidio with all 6 spaCy models (requires the optional downloads above)
- `skyflow` — Skyflow Detect API; `entity_types` auto-derived from dataset canonicals (requires creds)
- `skyflow_full` — Skyflow Detect API with all ~70 entity types (requires creds)

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

import torch
from opf_eval import fixtures, runner, report

# === EDIT THESE ===
DATASET = "pii_masking_200k"  # one of: pii_masking_300k, pii_masking_200k, pii_masking_400k, openpii_nano, openpii_mini
N_EXAMPLES = 100              # 100 for a quick smoke test, 1000 for a real bench, 5000 for stable signal
# Pick from any of these (eval-side runner names):
#   opf
#   gliner, gliner_nvidia, gliner_gretel_small, gliner_gretel_large
#   ai4privacy_modernbert, openmed
#   presidio, presidio_multilang
#   skyflow, skyflow_full        # add these after setting credentials below
DETECTORS = ["presidio", "gliner", "gliner_nvidia", "opf"]
FIXTURE_SEED = 42             # deterministic sample
RUN_NAME = "colab_demo"       # used as the output dir
# ===================

# Device: cuda > mps > cpu. To force CPU, set DEVICE = "cpu".
# Colab GPU runtime: Runtime -> Change runtime type -> T4 / L4 / A100 GPU.
# OPF speeds up ~10×, gliner_nvidia ~10×, others ~3-5× on T4 vs CPU.
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu"
)

FIXTURES_PATH = Path(f"/content/data/{DATASET}_{N_EXAMPLES}.jsonl")
OUT_DIR = Path(f"/content/results/{RUN_NAME}")

FIXTURES_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"device:                 {DEVICE}")
print(f"dataset:                {DATASET}")
print(f"will write fixtures to: {FIXTURES_PATH}")
print(f"will write results to:  {OUT_DIR}")
print(f"detectors:              {DETECTORS}")

## 3. (Optional) Skyflow credentials — skip if not benchmarking Skyflow

Add these to **Colab Secrets** (key icon in the left sidebar):
- `SKYFLOW_VAULT_URL` — e.g. `https://abc123.vault.skyflowapis.com`
- `SKYFLOW_VAULT_ID` — vault UUID
- `SKYFLOW_BEARER_TOKEN` — short-lived bearer token

Then run the cell below to expose them as env vars.

In [ ]:
try:
    from google.colab import userdata
    for var in ("SKYFLOW_VAULT_URL", "SKYFLOW_VAULT_ID", "SKYFLOW_BEARER_TOKEN"):
        try:
            os.environ[var] = userdata.get(var)
        except Exception:
            print(f"  {var}: not set in Colab Secrets")
    if all(v in os.environ for v in ("SKYFLOW_VAULT_URL", "SKYFLOW_VAULT_ID", "SKYFLOW_BEARER_TOKEN")):
        print("Skyflow creds loaded.")
    else:
        print("Skyflow creds incomplete — skip the skyflow detectors in DETECTORS.")
except ImportError:
    print("Not in Colab. Set SKYFLOW_VAULT_URL/SKYFLOW_VAULT_ID/SKYFLOW_BEARER_TOKEN in your shell env instead.")

## 4. Materialize the fixture set

Pulls a deterministic sample from PII-Masking-300k via HuggingFace Datasets, projects gold spans through the canonical taxonomy, writes JSONL.

First run downloads ~700 MB of dataset shards; cached for subsequent runs.

In [ ]:
if not FIXTURES_PATH.exists():
    n = fixtures.materialize(FIXTURES_PATH, N_EXAMPLES, dataset=DATASET, seed=FIXTURE_SEED)
    print(f"wrote {n} examples to {FIXTURES_PATH}")
else:
    print(f"reusing existing fixtures at {FIXTURES_PATH}")

import json
with FIXTURES_PATH.open() as f:
    sample = json.loads(f.readline())
print(f"\nfirst fixture keys: {list(sample.keys())}")
print(f"first fixture preview: {sample['text'][:120]}...")
print(f"first fixture gold spans: {len(sample['gold_spans'])} spans")

## 5. Run the detectors

Each detector's predictions are streamed to `raw_<detector>.jsonl` in `OUT_DIR`. Re-running is idempotent for already-completed detectors thanks to the manifest merge logic.

**Per-detector wall time** at 100 examples (rough; CPU on Colab free tier; GPU is ~5-10× faster for OPF + larger GLiNER variants):

| detector | CPU @ 100 | T4 GPU @ 100 |
|---|---|---|
| Presidio | ~30 s | n/a (CPU only) |
| ai4privacy_modernbert | ~30 s | ~10 s |
| gliner_gretel_small | ~45 s | ~15 s |
| GLiNER (default) | ~1 min | ~15 s (first run downloads ~500 MB) |
| gliner_nvidia / gliner_gretel_large | ~3-4 min | ~30 s (570M / 500M models) |
| OPF | ~1 min on short inputs, ~10 min at 1k | ~1 min at 1k (first run downloads ~2.8 GB) |
| OpenMed | ~1 min English, +30-60 s/lang | ~30 s English, +15-30 s/lang |
| Skyflow | ~2 min (network-bound, 1 req/s throttle) | n/a (HTTP) |

In [ ]:
runner.run(
    fixtures=FIXTURES_PATH,
    detector_names=DETECTORS,
    out_dir=OUT_DIR,
    dataset=DATASET,             # threads dataset_canonicals into skyflow + gliner builders
    device=DEVICE,               # cuda/mps/cpu — auto-detected in cell 5
    skyflow_workers=1,           # serial; bump if your Skyflow plan allows
    skyflow_min_interval_ms=0,   # add throttling if rate-limited
)

print("\nfiles written:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name}")

## 6. Generate and display the report

Builds the markdown report (headline + per-category + per-language) and renders it inline.

In [ ]:
md = report.build_report(OUT_DIR, FIXTURES_PATH)
(OUT_DIR / "report.md").write_text(md)

display(Markdown(md))

## 7. (Optional) Visualizations

Bar charts of per-category F1 and a latency comparison. Useful for slide decks.

In [ ]:
import json, statistics
import matplotlib.pyplot as plt
import numpy as np

from opf_eval.datasets import get as get_dataset_config
from opf_eval.taxonomy import dataset_canonicals
from opf_eval.nervaluate_metrics import score as semeval_score

fixture_records = [json.loads(l) for l in FIXTURES_PATH.open() if l.strip()]
fixture_index = {r["id"]: r for r in fixture_records}

# Use the chosen dataset's annotated canonicals as the chart x-axis.
vocab_key = get_dataset_config(DATASET).vocab_key
labels = sorted(dataset_canonicals(vocab_key))
print(f"chart labels ({len(labels)}): {labels}")

def detector_data(name):
    """Run nervaluate against this detector's raw output. Returns
    (per-tag F1 dict, latencies). Same Type-schema F1 as the report's
    per-category section."""
    path = OUT_DIR / f"raw_{name}.jsonl"
    if not path.exists():
        return None
    records = [json.loads(l) for l in path.open() if l.strip()]
    pairs = []
    latencies = []
    for r in records:
        if r.get("error"):
            continue
        gold = fixture_index[r["id"]]["gold_spans"]
        # Filter both sides to the dataset's annotated canonicals.
        pred = [s for s in r["spans"] if s["label"] in set(labels)]
        gold = [s for s in gold if s["label"] in set(labels)]
        pairs.append((pred, gold))
        latencies.append(r["latency_ms"])
    sem = semeval_score(detector=name, pairs=pairs, tags=labels)
    by_label_f1 = {
        lbl: sem.by_label.get(lbl, {}).get("ent_type", {}).get("f1", 0.0)
        for lbl in labels
    }
    return by_label_f1, latencies

# Auto-discover every detector with a raw_*.jsonl in OUT_DIR.
all_detectors = sorted(p.stem.removeprefix("raw_") for p in OUT_DIR.glob("raw_*.jsonl"))
results = {d: detector_data(d) for d in all_detectors if detector_data(d)}
print(f"detectors in this run: {list(results)}")

# === Per-category F1 bar chart (SemEval Type schema) ===
x = np.arange(len(labels))
bar_w = 0.8 / max(len(results), 1)
fig, ax = plt.subplots(figsize=(max(12, 0.9 * len(labels)), 5))
for i, (det, (f1s_dict, _)) in enumerate(results.items()):
    f1s = [f1s_dict.get(lbl, 0.0) for lbl in labels]
    ax.bar(x + i * bar_w, f1s, bar_w, label=det)
ax.set_xticks(x + bar_w * (len(results) - 1) / 2)
ax.set_xticklabels(labels, rotation=20)
ax.set_ylabel("SemEval Type F1 (any overlap + matching label)")
ax.set_title(f"Per-category F1 by detector ({DATASET})")
ax.set_ylim(0, 1)
ax.legend(loc="upper right", ncol=2 if len(results) > 4 else 1, fontsize=8)
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# === Latency comparison ===
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
p50s = [statistics.median(results[n][1]) for n in names]
p95s = [sorted(results[n][1])[int(0.95 * (len(results[n][1]) - 1))] for n in names]
p99s = [sorted(results[n][1])[int(0.99 * (len(results[n][1]) - 1))] for n in names]
x = np.arange(len(names))
ax.bar(x - 0.25, p50s, 0.25, label="p50")
ax.bar(x, p95s, 0.25, label="p95")
ax.bar(x + 0.25, p99s, 0.25, label="p99")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15)
ax.set_ylabel("latency (ms)")
ax.set_title("Per-detector latency")
ax.set_yscale("log")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5, which="both")
plt.tight_layout()
plt.show()

## 8. (Optional) Adding more detectors after the fact

Already ran OPF + GLiNER + Presidio and now want to add Skyflow without re-running the others? Set Skyflow creds (cell 3), then run the cell below. Re-run cell 15 afterward to refresh the charts — it auto-discovers any new `raw_<detector>.jsonl` in `OUT_DIR`.


In [ ]:
runner.run(
    fixtures=FIXTURES_PATH,
    detector_names=["skyflow"],   # only the new one
    out_dir=OUT_DIR,                # same dir
    dataset=DATASET,
    device=DEVICE,
)
md = report.build_report(OUT_DIR, FIXTURES_PATH)
(OUT_DIR / "report.md").write_text(md)
display(Markdown(md))


## 9. (Optional) Saving the run

Colab storage is ephemeral. To keep results, mount Drive and copy the run dir.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
dest = "/content/drive/MyDrive/pii_benchmark_runs/" + OUT_DIR.name
shutil.copytree(OUT_DIR, dest, dirs_exist_ok=True)
print(f"copied {OUT_DIR} -> {dest}")


# Sanitization

Detection (sections 1–9 above) tells you _where_ sensitive data is. Sanitization is what you do with it. The cells below show, side-by-side on the same fixtures, what each of the three available transform modes produces — from plain `[LABEL]` placeholders all the way to deterministic Skyflow vault tokens that survive across documents.


## 1. Configure the demo

The benchmark above scores *detection* quality (did the detector find the right spans?). The cells below answer a different question: **once you've found the spans, what does the transformed text actually look like?**

Four sanitization modes are demonstrated side-by-side on the same fixtures the detectors above ran against:

| Mode | Looks like | What it preserves |
|---|---|---|
| `redact` | `********` | Nothing — fixed-length asterisks regardless of original span length. |
| `label` | `[EMAIL]` | Category only. All Alices and all Bobs look identical. |
| `label_number` | `[EMAIL_1]` | Identity **within one document**: same value gets the same number; different values get different numbers. |
| `label_token` | `[EMAIL_jRc7QGn]` | Identity **across documents and time** via a Skyflow vault. Deterministic — the same plaintext maps to the same token forever, regardless of which detector found it. |

Edit the cell below to change the demo size or detector list. Sections 2–5 reuse the `raw_<detector>.jsonl` files from section 5 of the benchmark — no detector re-runs.


In [ ]:
# === EDIT THESE ===
DEMO_N_EXAMPLES = 6                         # how many fixtures to display (kept small — qualitative)
DEMO_DETECTORS = DETECTORS                  # reuses whatever cell 5 had
DEMO_MODES = ["redact", "label", "label_number", "label_token"]
# ===================

print(f"will render {DEMO_N_EXAMPLES} examples × {len(DEMO_DETECTORS)} detectors × {len(DEMO_MODES)} modes")
print(f"detectors: {DEMO_DETECTORS}")
print(f"modes:     {DEMO_MODES}")

## 2. (Optional) Token-vault credentials — required for `label_token` mode

The `label_token` mode inserts each detected entity into a Skyflow vault configured with deterministic format-preserving tokens (7-char alphanumeric), and uses the token Skyflow returns as the replacement. Skipping this cell leaves the column blank — the other two modes still work.

Operator one-time setup is documented in [docs/token-vault-setup.md](https://github.com/jstjoe/local-privacy/blob/main/docs/token-vault-setup.md). Once the vault exists, set three Colab secrets: `SKYFLOW_TOKEN_VAULT_URL`, `SKYFLOW_TOKEN_VAULT_ID`, and `SKYFLOW_TOKEN_BEARER_TOKEN` (or reuse `SKYFLOW_BEARER_TOKEN` from cell 7).

In [ ]:
import os
from opf_api.vault_tokens import TokenVaultClient

try:
    from google.colab import userdata  # type: ignore[import-not-found]
    for key in ("SKYFLOW_TOKEN_VAULT_URL", "SKYFLOW_TOKEN_VAULT_ID", "SKYFLOW_TOKEN_BEARER_TOKEN"):
        try:
            os.environ.setdefault(key, userdata.get(key) or "")
        except Exception:
            pass
except ImportError:
    pass  # not on Colab — env should already be set

TOKEN_VAULT = TokenVaultClient.from_env()
if TOKEN_VAULT is None:
    print("token vault not configured — `label_token` column will show a placeholder.")
    print("set SKYFLOW_TOKEN_VAULT_URL + _ID + a bearer to enable it.")
else:
    print("token vault configured — `label_token` column will show real 7-char tokens.")

## 3. Load fixtures and detector predictions

Re-read the fixture JSONL the runner wrote, and the per-detector `raw_<detector>.jsonl` files from section 5. We pick the first `DEMO_N_EXAMPLES` fixtures for the demo grid below — small on purpose, since this section is qualitative.


In [ ]:
# Load fixtures + each detector's predictions from disk.
import json

with FIXTURES_PATH.open() as f:
    all_fixtures = [json.loads(line) for line in f]

demo_fixtures = all_fixtures[:DEMO_N_EXAMPLES]

predictions = {}  # {detector_name: {fixture_id: [span, ...]}}
for detector in DEMO_DETECTORS:
    path = OUT_DIR / f"raw_{detector}.jsonl"
    if not path.exists():
        print(f"skipping {detector}: {path.name} not found — run cell 11 first")
        continue
    by_id = {}
    with path.open() as f:
        for line in f:
            row = json.loads(line)
            by_id[row["id"]] = row.get("spans") or []
    predictions[detector] = by_id

print(f"loaded predictions for: {sorted(predictions)}")
print(f"first demo fixture id: {demo_fixtures[0]['id']}, text preview: {demo_fixtures[0]['text'][:80]}...")

## 4. Render the side-by-side grid

For each example, render every detector's predictions in every transform mode. The result is one markdown table per example: rows are detectors, columns are modes. Look across a row to compare what each mode does to the same detected spans; look down a column to compare detectors at one mode.


In [ ]:
from IPython.display import Markdown, display
from opf_eval.transforms import render_modes

def _esc(s: str) -> str:
    return s.replace("|", "\\|").replace("\n", " ")

for fx in demo_fixtures:
    fid = fx["id"]
    text = fx["text"]
    parts = [f"### Example `{fid}`\n", f"> {_esc(text)}\n"]
    header = "| detector | " + " | ".join(DEMO_MODES) + " |"
    sep = "|" + "|".join(["---"] * (len(DEMO_MODES) + 1)) + "|"
    parts.append(header)
    parts.append(sep)
    for detector, by_id in predictions.items():
        spans = by_id.get(fid, [])
        rendered = render_modes(
            text, spans,
            modes=DEMO_MODES,
            detector_name=detector,
            token_vault_client=TOKEN_VAULT,
        )
        row = [detector] + [_esc(rendered.get(m, "")) for m in DEMO_MODES]
        parts.append("| " + " | ".join(row) + " |")
    display(Markdown("\n".join(parts)))

**Reading the rows.**

- Identical label and label_number columns mean the detector found only one entity in that example.
- `label_token` shows a `(set SKYFLOW_TOKEN_VAULT_* to enable)` placeholder when the vault from section 2 above is unconfigured. Spans whose canonical label has no vault column (rare) fall back to `[LABEL]` per span.
- Different detectors disagree on span boundaries and labels — that's the same disagreement the F1 scores in section 6 quantify, surfaced here visually.


## 5. Determinism check across documents

`label_token` is the only mode whose output stays consistent across completely separate documents. This cell runs a chosen detector live against a small set of texts that intentionally share a few PII values, tokenizes each, and prints a summary calling out every plaintext that mapped to the same token in 2+ inputs. Edit `DEMO_DETERMINISM_TEXTS` to drop in your own sample.


In [ ]:
# Cross-document determinism check.
#
# label_token is the only mode that keeps PII identity stable across
# completely separate documents. To make this easy to see, we run a
# detector live against a handful of sample texts that *intentionally*
# share a few PII values, then tokenize and confirm the same plaintext
# always gets the same 7-char token.
#
# Edit DEMO_DETERMINISM_TEXTS to drop in your own sample. The default
# repeats `alice@example.com` three times across three texts.

DEMO_DETERMINISM_TEXTS = [
    "Email Alice at alice@example.com about the quarterly report. She replied from alice@example.com later that day.",
    "Bob followed up with alice@example.com and cc'd jay@example.com on the thread.",
    "If you can't reach Alice, try +1-415-555-0100 or alice@example.com.",
]

# Which detector to run live against the texts above. None = the first
# detector in DEMO_DETECTORS that the notebook actually loaded; presidio
# is preferred because it's fast and CPU-only — other detectors may be
# slower on first call.
DEMO_DETERMINISM_DETECTOR = None

if TOKEN_VAULT is None:
    print("vault not configured — skipping determinism check.")
else:
    # _build_detector is technically a private entry point; using it here
    # keeps the demo self-contained (one call constructs whichever detector
    # the user picked with the same wiring runner.run uses internally).
    from opf_eval.runner import _build_detector
    from opf_eval.taxonomy import CANONICAL_LABELS
    from opf_eval.transforms import splice_spans, label_token_renderer

    det_name = DEMO_DETERMINISM_DETECTOR
    if det_name is None:
        if "presidio" in DEMO_DETECTORS:
            det_name = "presidio"
        elif DEMO_DETECTORS:
            det_name = DEMO_DETECTORS[0]
        else:
            det_name = "presidio"

    print(f"running live detection with: {det_name}\n")
    det = _build_detector(
        det_name,
        dataset_canonicals_set=set(CANONICAL_LABELS),
        device=DEVICE,
    )

    tokens_by_kv = {}     # (label, value) -> token
    seen_in_texts = {}    # (label, value) -> set of text indices
    rendered_rows = []    # list of (i, original_text, label_token_text)

    for i, text in enumerate(DEMO_DETERMINISM_TEXTS, start=1):
        spans = det.detect(text).get("spans") or []
        # Build the vault renderer once per text — one batch insert. Same
        # plaintext returns the same token across calls, so reuse-by-call
        # is cheap even if a value appears in multiple texts.
        vrender = label_token_renderer(spans, TOKEN_VAULT)
        vault_text = splice_spans(text, spans, vrender)
        rendered_rows.append((i, text, vault_text))

        for s in spans:
            label = s["label"]
            value = s["text"]
            key = (label, value)
            # Ask the renderer directly for THIS span's token — robust
            # against multiple bracketed tokens in the spliced text.
            rendered = vrender(s)
            prefix = f"[{label}_"
            if not (rendered.startswith(prefix) and rendered.endswith("]")):
                continue  # fallback like "[EMAIL]" — no per-value token
            tok = rendered[len(prefix):-1]
            if key in tokens_by_kv:
                assert tokens_by_kv[key] == tok, (
                    f"label_token NOT deterministic for {key}: "
                    f"saw {tokens_by_kv[key]} then {tok}"
                )
            tokens_by_kv[key] = tok
            seen_in_texts.setdefault(key, set()).add(i)

    # Render side-by-side: each input next to its label_token output.
    parts = ["### Inputs and `label_token` outputs\n"]
    for i, text, vt in rendered_rows:
        parts.append(f"**Text {i}**")
        parts.append(f"> {text}")
        parts.append(f"> {vt}")
        parts.append("")
    display(Markdown("\n\n".join(parts)))

    # Token table — one row per unique (label, value) the detector found,
    # with how many *documents* (not raw detections) it showed up in.
    if tokens_by_kv:
        rows = ["### Token table", "", "| label | plaintext | token | documents |", "|---|---|---|---|"]
        for (label, plain), tok in sorted(tokens_by_kv.items()):
            n = len(seen_in_texts[(label, plain)])
            rows.append(f"| `{label}` | `{plain}` | `[{label}_{tok}]` | {n} |")
        display(Markdown("\n".join(rows)))

    # Highlight the cross-document property — values whose plaintext
    # appeared in 2+ distinct input texts. Counting documents (not raw
    # span detections) avoids falsely flagging within-text duplicates.
    repeated = [
        (k, tokens_by_kv[k], len(seen_in_texts[k]))
        for k in tokens_by_kv
        if len(seen_in_texts[k]) >= 2
    ]
    if repeated:
        print("\n✓ deterministic across documents — these values appeared in multiple inputs and got the same token every time:")
        for (label, plain), tok, n in repeated:
            print(f"    {label}={plain!r}  →  [{label}_{tok}]   (seen in {n} documents)")
    else:
        print("\n(no value appeared in 2+ documents — edit DEMO_DETERMINISM_TEXTS above to add a value that recurs across texts)")
